# Yahoo OHLCV audit

This notebook audits the forecasting data currently stored in the repository and re-downloads daily OHLCV data from Yahoo Finance for the same 23 assets.

The goal is to verify which preprocessing techniques from the financial data preprocessing workshop can be applied to this dataset.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "util.py").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "util.py").exists())

sys.path.insert(0, str(PROJECT_ROOT / "model" / "preprocessing"))

import pandas as pd

from preprocessing_utils import (
    DATA_OUT,
    get_forecasting_tickers,
    get_forecasting_date_range,
    download_yahoo_ohlcv,
    extract_field,
    compute_universe_activity,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_OUT:", DATA_OUT)


## Existing repository data

The existing forecasting data is based on adjusted close prices and log returns.

In [ ]:
prices = pd.read_parquet(PROJECT_ROOT / "data" / "precios_close.parquet")
returns = pd.read_parquet(PROJECT_ROOT / "data" / "returns.parquet")

audit_rows = [
    {
        "dataset": "precios_close",
        "shape": str(prices.shape),
        "start_date": prices.index.min(),
        "end_date": prices.index.max(),
        "n_assets": prices.shape[1],
        "columns_available": "Close only",
    },
    {
        "dataset": "returns",
        "shape": str(returns.shape),
        "start_date": returns.index.min(),
        "end_date": returns.index.max(),
        "n_assets": returns.shape[1],
        "columns_available": "Log returns only",
    },
]

audit = pd.DataFrame(audit_rows)
display(audit)

print("Tickers:")
print(list(prices.columns))


## Re-download Yahoo OHLCV

Yahoo Finance provides daily Open, High, Low, Close and Volume. This is enough to build daily count, volume and dollar bars, but not transaction-level tick bars.

In [ ]:
tickers = get_forecasting_tickers()
start, end = get_forecasting_date_range()

# yfinance treats end as exclusive, so add one day.
start_str = start.strftime("%Y-%m-%d")
end_str = (end + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

print("Downloading:", len(tickers), "tickers")
print("Range:", start_str, "->", end_str)

ohlcv_path = DATA_OUT / "yahoo_ohlcv.parquet"

ohlcv = download_yahoo_ohlcv(
    tickers=tickers,
    start=start_str,
    end=end_str,
    output_path=ohlcv_path,
)

print("Saved:", ohlcv_path)
print("Shape:", ohlcv.shape)
display(ohlcv.head())


## OHLCV availability check

In [ ]:
close = extract_field(ohlcv, "Close")
volume = extract_field(ohlcv, "Volume")
activity = compute_universe_activity(ohlcv)

availability = pd.DataFrame({
    "field": ["Close", "Volume"],
    "shape": [str(close.shape), str(volume.shape)],
    "missing_values": [int(close.isna().sum().sum()), int(volume.isna().sum().sum())],
    "start_date": [close.index.min(), volume.index.min()],
    "end_date": [close.index.max(), volume.index.max()],
})

availability_path = DATA_OUT / "yahoo_ohlcv_audit_summary.csv"
availability.to_csv(availability_path, index=False)

activity_path = DATA_OUT / "universe_daily_activity.csv"
activity.to_csv(activity_path)

print("Saved:", availability_path)
print("Saved:", activity_path)
display(availability)
display(activity.head())


## Conclusion

The saved forecasting files only contain close prices and returns. After re-downloading daily OHLCV from Yahoo Finance, we can construct daily count bars, volume bars and dollar bars using the aggregated universe activity.

Real tick bars cannot be constructed because Yahoo Finance does not provide transaction-level trades in this dataset.